In [1]:
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

Dynamically set JAVA_HOME: /Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home
HOMEBREW_PREFIX: /opt/homebrew
COMMAND_MODE: unix2003
INFOPATH: /opt/homebrew/share/info:
SHELL: /bin/zsh
PYTHONPATH: /Users/alexandra.miron/PycharmProjects/data4team21:/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev:/Applications/PyCharm.app/Contents/plugins/python/helpers-pro/jupyter_debug
__CFBundleIdentifier: com.jetbrains.pycharm
TMPDIR: /var/folders/nh/bnybds3d02s2103s2s3xkdyw0000gp/T/
LC_ALL: en_US.UTF-8
HOME: /Users/alexandra.miron
HOMEBREW_REPOSITORY: /opt/homebrew
PATH: /Users/alexandra.miron/PyCharmMiscProject/.venv/bin:/opt/homebrew/bin:/opt/homebrew/sbin:/Users/alexandra.miron/tools/spark-3.5.2-bin-hadoop3/bin:/Users/alexandra.miron/tools/hadoop-3.4.0-win10-x64/bin:/Library/Frameworks/Python.framework/Versions/3.11/bin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var

In [3]:
spark = cc.startLocalCluster("dimVehicle")
spark.getActiveSession()

In [4]:
# EXTRACT
cc.set_connectionProfile("velodb")

df_vehicles = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "vehicles") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_bikelots = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bikelots") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_biketypes = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bike_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

In [5]:
print(cc.create_jdbc())

jdbc:postgresql://localhost:5432/velodb


In [6]:
# TRANSFORM: join vehicles → bikelots → bike_types
df_vehicle_dim = df_vehicles.alias("v") \
    .join(df_bikelots.alias("b"), col("v.bikelotid") == col("b.bikelotid"), "left") \
    .join(df_biketypes.alias("t"), col("b.biketypeid") == col("t.biketypeid"), "left") \
    .select(
        col("v.vehicleid").alias("vehicle_id"),
        col("t.biketypedescription").alias("type")
    )

In [7]:
# Inspect schema and preview
df_vehicle_dim.show()
df_vehicle_dim.printSchema()

# LOAD: write as delta table
df_vehicle_dim.write.format("delta").mode("overwrite").saveAsTable("VehicleDim")

+----------+---------+
|vehicle_id|     type|
+----------+---------+
|         1|Velo Bike|
|         2|Velo Bike|
|         3|Velo Bike|
|         4|Velo Bike|
|         5|Velo Bike|
|         6|Velo Bike|
|         7|Velo Bike|
|         8|Velo Bike|
|         9|Velo Bike|
|        10|Velo Bike|
|        11|Velo Bike|
|        12|Velo Bike|
|        13|Velo Bike|
|        14|Velo Bike|
|        15|Velo Bike|
|        16|Velo Bike|
|        17|Velo Bike|
|        18|Velo Bike|
|        19|Velo Bike|
|        20|Velo Bike|
+----------+---------+
only showing top 20 rows

root
 |-- vehicle_id: integer (nullable = true)
 |-- type: string (nullable = true)



25/05/05 19:46:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Delete the spark session

In [9]:
spark.stop()